# v3.0.0 Depression AST Spectrograms (Supplementary)

Depression screening on v3.0.0. Results for supplementary only (44 cases).
Tasks: `animal-fluency`, `open-response-questions` (if available), or fallback to tasks with depression coverage.

In [ ]:
#1 - Imports & paths
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from tqdm import tqdm

ROOT = Path('/data0/b2ai-voice/3.0.0')
SPEC = ROOT / 'features' / 'torchaudio_mel_spectrogram.parquet'
DEP_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'depression.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')

# Load spectrogram data
pf = pq.ParquetFile(SPEC)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','mel_spectrogram','n_frames']).to_pandas())
spec = pd.concat(parts, ignore_index=True)
spec['participant_id'] = spec['participant_id'].astype(str).str.zfill(6)
print(f'Total recordings: {len(spec)}')

In [ ]:
#2 - Build depression labels and find available tasks
dep_df = pd.read_csv(DEP_PHEN, sep='\t')
ctrl_df = pd.read_csv(CTRL_PHEN, sep='\t')
dep_ids = set(dep_df['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_df['participant_id'].astype(str).str.zfill(6)) - dep_ids

spec['label'] = np.nan
spec.loc[spec['participant_id'].isin(dep_ids), 'label'] = 1
spec.loc[spec['participant_id'].isin(ctrl_ids), 'label'] = 0
data = spec.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)

print(f'Depression IDs: {len(dep_ids)}, Control IDs: {len(ctrl_ids)}')

# Find tasks with best depression + control coverage
task_coverage = []
for task in data['task_name'].unique():
    task_data = data[data['task_name'] == task]
    dep_ppl = task_data[task_data['label']==1]['participant_id'].nunique()
    ctrl_ppl = task_data[task_data['label']==0]['participant_id'].nunique()
    if dep_ppl > 0 and ctrl_ppl > 0:
        task_coverage.append((task, dep_ppl, ctrl_ppl, dep_ppl + ctrl_ppl))

task_coverage.sort(key=lambda x: x[3], reverse=True)
print('\nTop tasks by depression+control coverage:')
for t, d, c, tot in task_coverage[:15]:
    print(f'  {t}: Dep={d}, Ctrl={c}, Total={tot}')

In [ ]:
#3 - Select tasks with best coverage and filter
# Use top tasks that have adequate depression and control representation
# Prioritize tasks available in v2.0.1 depression analysis: animal-fluency
# Plus additional tasks with good coverage
SELECTED_TASKS = [t[0] for t in task_coverage[:8]]  # Top 8 tasks by coverage
MIN_TIME_FRAMES = 100

data_selected = data[
    (data['task_name'].isin(SELECTED_TASKS)) &
    (data['n_frames'] >= MIN_TIME_FRAMES)
].copy()

print(f'Selected tasks: {SELECTED_TASKS}')
print(f'Recordings: {len(data_selected)}')
print(f'Depression: {data_selected[data_selected["label"]==1]["participant_id"].nunique()} ppl')
print(f'Control: {data_selected[data_selected["label"]==0]["participant_id"].nunique()} ppl')

In [ ]:
#4 - Process spectrograms
from scipy.ndimage import zoom

TARGET_SEQ_LEN = 1024

def process_spectrogram(spec_raw, target_len=1024):
    spec = np.stack(spec_raw).astype(np.float32)
    n_mels, time_len = spec.shape
    if time_len < target_len:
        spec = np.pad(spec, ((0, 0), (0, target_len - time_len)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        spec = spec[:, start:start + target_len]
    return spec

def resize_spectrogram(spec, target_mel=128, target_time=1024):
    mel_ratio = target_mel / spec.shape[0]
    time_ratio = target_time / spec.shape[1]
    return zoom(spec, (mel_ratio, time_ratio), order=1).astype(np.float32)

X_list = []
for idx, row in tqdm(data_selected.iterrows(), total=len(data_selected), desc='Processing'):
    X_list.append(process_spectrogram(row['mel_spectrogram'], TARGET_SEQ_LEN))

X_raw = np.stack(X_list)
y_raw = data_selected['label'].values
participants_raw = data_selected['participant_id'].values
print(f'Processed shape: {X_raw.shape}')

In [ ]:
#5 - Model and 5-Fold CV (identical pipeline to PD)
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
from sklearn.metrics import confusion_matrix, accuracy_score
from scipy import stats
import copy, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_base=False):
        super().__init__()
        if pretrained:
            self.ast = ASTModel.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
            hidden_size = self.ast.config.hidden_size
        else:
            config = ASTConfig(hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                             intermediate_size=3072, max_length=1024, num_mel_bins=128)
            self.ast = ASTModel(config); hidden_size = 768
        if freeze_base:
            for param in self.ast.parameters(): param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 256),
            nn.GELU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
    def forward(self, x):
        x = x.transpose(1, 2)
        return self.classifier(self.ast(input_values=x).pooler_output)

class ASTDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150); t0 = np.random.randint(0, max(1, x.shape[1]-t)); x[:, t0:t0+t] = 0
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30); f0 = np.random.randint(0, max(1, x.shape[0]-f)); x[f0:f0+f, :] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1-pt)**self.gamma)*ce).mean()

def evaluate_fold(model, loader, device):
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            probs = torch.softmax(model(batch['inputs'].to(device)), dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs); all_labels.extend(batch['labels'].numpy()); all_parts.extend(batch['participant'])
    all_probs, all_labels, all_parts = np.array(all_probs), np.array(all_labels), np.array(all_parts)
    unique_parts = np.unique(all_parts)
    return (np.array([all_probs[all_parts==p].mean() for p in unique_parts]),
            np.array([all_labels[all_parts==p][0] for p in unique_parts]), unique_parts)

# CV
unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])
print(f'Participants: {len(unique_participants)}, Depression: {int(participant_labels.sum())}, Control: {int((participant_labels==0).sum())}')

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
all_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
all_oof_labels = participant_labels.astype(np.int64).copy()
norm_stats = {}
total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    train_parts_fold = unique_participants[train_idx]; val_parts_fold = unique_participants[val_idx]
    train_mask = np.isin(participants_raw, train_parts_fold); val_mask = np.isin(participants_raw, val_parts_fold)
    X_train_fold = X_raw[train_mask]; y_train_fold = y_raw[train_mask]; parts_train = participants_raw[train_mask]
    X_val_fold = X_raw[val_mask]; y_val_fold = y_raw[val_mask]; parts_val = participants_raw[val_mask]
    print(f'Train: {len(X_train_fold)} recs from {len(train_parts_fold)} ppl')

    X_tr_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_train_fold, desc='resize', leave=False)])
    X_va_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_val_fold, desc='resize', leave=False)])
    fold_mean, fold_std = X_tr_ast.mean(), X_tr_ast.std()
    X_tr_ast = (X_tr_ast - fold_mean) / (fold_std + 1e-8)
    X_va_ast = (X_va_ast - fold_mean) / (fold_std + 1e-8)
    norm_stats[fold+1] = {'mean': float(fold_mean), 'std': float(fold_std)}

    train_ds = ASTDataset(X_tr_ast, y_train_fold, parts_train, augment=True)
    val_ds = ASTDataset(X_va_ast, y_val_fold, parts_val, augment=False)
    cc = np.bincount(y_train_fold); sw = (1.0/cc)[y_train_fold]
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=torch.utils.data.WeightedRandomSampler(sw, len(sw)), num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)
    bp = [p for n,p in model.named_parameters() if 'classifier' not in n]
    hp = [p for n,p in model.named_parameters() if 'classifier' in n]
    optimizer = torch.optim.AdamW([{'params': bp, 'lr': 5e-6, 'weight_decay': 0.01}, {'params': hp, 'lr': 5e-4, 'weight_decay': 0.01}], betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)
    cw = (cc.sum() / (2.0*cc)).astype(np.float32)
    criterion = FocalLoss(alpha=torch.tensor(cw, dtype=torch.float32).to(device), gamma=2.0)

    best_score, best_state, pat = 0, None, 0
    for epoch in range(30):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(); loss = criterion(model(batch['inputs'].to(device)), batch['labels'].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        scheduler.step()
        pp, pl, _ = evaluate_fold(model, val_loader, device)
        if len(np.unique(pl)) > 1:
            auc = roc_auc_score(pl, pp); fpr,tpr,thr = roc_curve(pl, pp)
            f1_opt = f1_score(pl, (pp >= thr[np.argmax(tpr-fpr)]).astype(int), zero_division=0)
        else: auc, f1_opt = 0.5, 0.0
        score = 0.4*auc + 0.6*f1_opt
        if score > best_score + 0.01: best_score = score; best_state = copy.deepcopy(model.state_dict()); pat = 0
        else: pat += 1
        if pat >= 10: break

    model.load_state_dict(best_state)
    pp, pl, vpids = evaluate_fold(model, val_loader, device)
    torch.save(model.state_dict(), str(RESULTS_DIR / f'ast_depression_v3_fold{fold+1}.pt'))
    for i, pid in enumerate(vpids): all_oof_probs[np.where(unique_participants==pid)[0][0]] = pp[i]
    fold_auc = roc_auc_score(pl, pp); fpr,tpr,thr = roc_curve(pl, pp)
    ot = thr[np.argmax(tpr-fpr)]; po = (pp >= ot).astype(int)
    fold_results.append({'fold': fold+1, 'auc': float(fold_auc),
        'f1_opt': float(f1_score(pl, po, zero_division=0)),
        'recall_opt': float(recall_score(pl, po, zero_division=0)),
        'precision_opt': float(precision_score(pl, po, zero_division=0)), 'threshold': float(ot)})
    print(f'Fold {fold+1}: AUC={fold_auc:.4f}')
    del model; torch.cuda.empty_cache()

print(f'\nTotal: {(time.time()-total_start)/60:.1f} min')

In [ ]:
#6 - Summary
aucs = [r['auc'] for r in fold_results]
t_crit = stats.t.ppf(0.975, df=4)
for name, arr in [('AUC', aucs), ('F1', [r['f1_opt'] for r in fold_results])]:
    m, sd = np.mean(arr), np.std(arr, ddof=1)
    print(f'{name}: {m:.4f} ({sd:.4f}) [{m-t_crit*sd/np.sqrt(5):.4f}, {m+t_crit*sd/np.sqrt(5):.4f}]')

oof_auc = roc_auc_score(all_oof_labels, all_oof_probs)
fpr,tpr,thr = roc_curve(all_oof_labels, all_oof_probs)
oof_thresh = thr[np.argmax(tpr-fpr)]
oof_preds = (all_oof_probs >= oof_thresh).astype(int)
print(f'\nOOF AUC: {oof_auc:.4f}  F1: {f1_score(all_oof_labels, oof_preds, zero_division=0):.4f}')

np.savez(str(RESULTS_DIR / 'ast_depression_v3_cv_results.npz'),
    oof_probs=all_oof_probs, oof_labels=all_oof_labels, participant_ids=unique_participants,
    fold_aucs=np.array(aucs), oof_auc=np.array(oof_auc))
print(f'Saved: {RESULTS_DIR}/ast_depression_v3_cv_results.npz')